In [1]:
# Import required libraries
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

# Aggregate Fluorescence Intensity Data for hqno reporter
## Overview
This notebook aggregates single-cell property data from multiple experimental replicates and positions for the different PA partner strains (WT and deficient in either HQNO and or RHL production)

## Workflow
1. **Input Processing**: Iterates through a predefined `data_list` of datasets, each specifying a source directory, subdirectory name, and strain label. Data comes from two source directories: `SAReporterGradient` and `SAReporterCallibration`.
2. **Data Collection**: For each dataset, searches for `replicate*` folders, then `pos*` folders within each replicate, and loads `single_cell_props.csv` when present.
3. **Metadata Addition**: Adds the following metadata columns to each dataframe:
   - `strain`: Strain label from `data_list` (one of `∆pqsL`, `∆pqsL ∆rhlA`, `wt`, `∆pqsL-old`)
   - `replicate`: Replicate identifier
   - `pos`: Position identifier
4. **Data Combination**: Concatenates all individual dataframes into a single unified dataset
5. **Output**: Saves the combined dataframe as `0_combined_df_hqno.csv`

In [ ]:
# Define input and output paths
# set path to BioImageArchive data directory:
data_archive_path = Path('/Volumes/ScientificData/Users/Giulia(botgiu00)/Papers/bottacin2026/BioImageArchive/')

# relative paths to data
input_dir1 =  data_archive_path / 'SAReporterGradient'  
input_dir2 =  data_archive_path / 'SAReporterCallibration' 

output_dir = './0_combined_df_hqno_legacy.csv'

data_list = [(input_dir1, 'S11_DpqsL_HQNO-reporter', '∆pqsL'),
 (input_dir1, 'S11_DrhlA-DpqsL_HQNO-reporter', '∆pqsL ∆rhlA'),
 (input_dir1, 'S11_wt_HQNO-reporter', 'wt-old'),
 (input_dir2, 'S12_DpqsL_HQNO-0ng', '∆pqsL-old')]


In [3]:

# Initialize list to store individual dataframes
all_data = []

# iterate through all datasets
for dataset in data_list:

    input_dir = dataset[0] / dataset[1]
    strain_name = dataset[2]

    if not os.path.isdir(input_dir):
        raise FileNotFoundError(
            f"Could not find input directory: {input_dir}. Current working dir: {os.getcwd()}"
        )

    # Iterate through replicate folders
    for replicate_folder in os.listdir(input_dir):
        replicate_path = os.path.join(input_dir, replicate_folder)

        # Process only replicate directories
        if os.path.isdir(replicate_path) and replicate_folder.startswith('replicate'):

            # Iterate through position folders within each replicate
            for pos_folder in os.listdir(replicate_path):
                if pos_folder.startswith('pos'):
                    pos_path = os.path.join(replicate_path, pos_folder, 'single_cell_props.csv')

                    # Load data if CSV file exists
                    if os.path.isfile(pos_path):
                        df = pd.read_csv(pos_path)

                        # Add metadata columns
                        df.insert(1, 'strain', strain_name)
                        df.insert(2, 'replicate', replicate_folder)
                        df.insert(3, 'pos', pos_folder)

                        # Append to collection
                        all_data.append(df)

# Combine all dataframes and save to output file
if all_data:
    combined_df_hqno = pd.concat(all_data, ignore_index=True)
    combined_df_hqno.to_csv(output_dir, index=False)
    print(f"Successfully combined {len(all_data)} datasets")
    print(f"Total rows: {len(combined_df_hqno)}")
    print(f"Output saved to: {output_dir}")

    # Display first few rows
    combined_df_hqno.head()
else:
    print("Warning: No data files found")

Successfully combined 74 datasets
Total rows: 257025
Output saved to: ./0_combined_df_hqno.csv


In [4]:
combined_df_hqno['strain'].unique()

array(['∆pqsL', '∆pqsL ∆rhlA', 'wt-old', '∆pqsL-old'], dtype=object)